# Predicting Spotify Streams with a basic ANN

Dataset: [Top Spotify Songs 2023](https://www.kaggle.com/datasets/nelgiriyewithana/top-spotify-songs-2023)

Goal: build a minimal fully-connected feed-forward neural network in PyTorch that predicts the `streams` column from numeric features.

Constraints (on purpose, for learning):
- No RNN / CNN — only `nn.Linear` + ReLU.
- No anti-overfitting tricks: no dropout, no weight decay, no batch norm, no early stopping. We *want* to see the model overfit so we can fix it later.
- Raw `streams` as the target (no log / standardize). The MSE values will look enormous — that is intentional and instructive.

## 1. Install & import

In [ ]:
!pip install kagglehub --quiet

In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 2. Download the dataset

`kagglehub` handles authentication automatically in Colab — no `kaggle.json` needed.

In [ ]:
path = kagglehub.dataset_download("nelgiriyewithana/top-spotify-songs-2023")
print("Dataset folder:", path)

df = pd.read_csv(f"{path}/spotify-2023.csv", encoding="latin-1")
df.head()

## 3. Quick EDA

Look at shape, dtypes, missing values, and the spread of the target.

In [ ]:
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nMissing values per column:\n", df.isna().sum())

In [ ]:
df['streams'] = pd.to_numeric(df['streams'], errors='coerce')
df['streams'].describe()

Notice `streams` ranges from a few thousand up to billions. We are deliberately keeping these raw; expect MSE to look huge.

## 4. Preprocessing

1. Drop rows where `streams` is invalid.
2. Drop non-numeric / high-cardinality columns (`track_name`, `artist(s)_name`, `key`, `mode`).
3. Coerce the playlist/chart columns to numeric — a few rows store them as strings.
4. Drop any remaining NaN rows.
5. Standardize features (this is for *optimization stability*, not regularization — it's fine under our constraints).

In [ ]:
data = df.copy()

data = data.dropna(subset=['streams'])

drop_cols = ['track_name', 'artist(s)_name', 'key', 'mode']
data = data.drop(columns=drop_cols)

for col in data.columns:
    data[col] = pd.to_numeric(data[col], errors='coerce')

data = data.dropna()

print("Cleaned shape:", data.shape)
data.head()

In [ ]:
y = data['streams'].values.astype(np.float32)
X = data.drop(columns=['streams']).values.astype(np.float32)

feature_names = data.drop(columns=['streams']).columns.tolist()
print("Number of features:", X.shape[1])
print("Features:", feature_names)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

print("X_train:", X_train.shape, "X_test:", X_test.shape)

## 5. Tensors & DataLoaders

In [ ]:
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

train_ds = TensorDataset(X_train_t, y_train_t)
test_ds = TensorDataset(X_test_t, y_test_t)

BATCH_SIZE = 32
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

## 6. Model definition

Three hidden layers, ReLU between them, a single linear output for regression.

In [ ]:
class StreamsANN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

input_dim = X_train.shape[1]
model = StreamsANN(input_dim).to(device)
print(model)

## 7. Loss & optimizer

MSE for regression. Adam with default `weight_decay=0` (any non-zero value would be L2 regularization, which we are skipping).

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

## 8. Training loop

In [ ]:
EPOCHS = 100
train_losses = []
test_losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_train_loss = 0.0
    n_train = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item() * xb.size(0)
        n_train += xb.size(0)
    epoch_train_loss /= n_train

    model.eval()
    epoch_test_loss = 0.0
    n_test = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            epoch_test_loss += loss.item() * xb.size(0)
            n_test += xb.size(0)
    epoch_test_loss /= n_test

    train_losses.append(epoch_train_loss)
    test_losses.append(epoch_test_loss)

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:3d} | train MSE: {epoch_train_loss:.3e} | test MSE: {epoch_test_loss:.3e}")

## 9. Evaluate on the test set

Compute MSE, RMSE (interpretable as average error in stream count), MAE, and R².

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(X_test_t.to(device)).cpu().numpy()

actual = y_test

mse = np.mean((preds - actual) ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(preds - actual))
ss_res = np.sum((actual - preds) ** 2)
ss_tot = np.sum((actual - actual.mean()) ** 2)
r2 = 1 - ss_res / ss_tot

print(f"MSE : {mse:.3e}")
print(f"RMSE: {rmse:.3e}  (avg error in streams)")
print(f"MAE : {mae:.3e}")
print(f"R^2 : {r2:.4f}")
print(f"\nFor reference, std(streams) on test set: {actual.std():.3e}")

## 10. Plots

Loss curves let you spot overfitting (train loss falling while test loss flattens or rises). The predicted-vs-actual scatter shows where the model is right and where it misses.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label='train MSE')
plt.plot(test_losses, label='test MSE')
plt.xlabel('Epoch')
plt.ylabel('MSE (raw streams)')
plt.yscale('log')
plt.title('Training vs test loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
plt.figure(figsize=(6, 6))
plt.scatter(actual, preds, alpha=0.5)
lims = [min(actual.min(), preds.min()), max(actual.max(), preds.max())]
plt.plot(lims, lims, 'r--', label='perfect prediction')
plt.xlabel('Actual streams')
plt.ylabel('Predicted streams')
plt.title('Predicted vs actual')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Next experiments to try

When you are ready to push past this baseline, consider one change at a time:

1. Replace the target with `np.log1p(y)` for training, then `np.expm1(pred)` at evaluation. The loss numbers and R² will become much friendlier.
2. Add `nn.Dropout(p=0.2)` after each ReLU.
3. Pass `weight_decay=1e-4` to `Adam` (L2 regularization).
4. Add an early-stopping check on `test_losses`.
5. Bring back `key` and `mode` via one-hot encoding, or build embeddings for `artist(s)_name`.